In [11]:
## Make perturbation experiments
import sys
sys.path.insert(0,"/g/data/gb02/ab8992/regional-mom6/dev/mom6_forge")
sys.path.insert(0,"/g/data/gb02/ab8992/regional-mom6/dev/regional-mom6")
import warnings
warnings.filterwarnings("ignore")
import xarray as xr

import os
from pathlib import Path
import regional_mom6 as rmom6


def IC_from_restart(basepath,expt,restart,out_path,start_date):
    rundir = basepath / expt
    restart = xr.open_mfdataset(str(basepath / expt / "archive" / restart / "access-om3.mom6.r.*"))[["Temp","Salt","u","v","sfc"]].isel(Time = 0)
    restart.to_netcdf(out_path / "raw_IC_data.nc",mode = "w")

    ## Put links to the hgrid and vgrid in the out_path so we can treat that as our run directory
    (out_path / "hgrid.nc").symlink_to(basepath / expt / "inputdir" / "hgrid.nc")
    (out_path / "vcoord.nc").symlink_to(basepath / expt / "inputdir" / "vcoord.nc")

    expt = rmom6.experiment(              
        date_range = [start_date, "2013-01-30"],
        hgrid_type="from_file",
        vgrid_type="from_file",
        tidal_constituents=[],
        mom_run_dir = "na",
        mom_input_dir = out_path,
    )

    ocean_varnames = {"time": "time",
                  "yh": "lath", 
                  "xh": "lonh",
                  "yq": "latq", 
                  "xq": "lonq",                  
                  "zl": "Layer",
                  "eta": "sfc",
                  "u": "u",
                  "v": "v",
                  "tracers": {"salt": "Salt", "temp": "Temp"} # You can have as many tracers here as you like, and they will all get regridded to your ICs / BCs
                  }

    # Set up the initial condition.
    expt.setup_initial_condition(
        expt.mom_input_dir / "raw_IC_data.nc",
        ocean_varnames,
        arakawa_grid = "C"
        )

    return

basepath = Path("/g/data/gb02/ab8992/rom3-rundirs/")
expt = "nino-12th"
out_path = Path("/g/data/x77/ab8992/nino-34/restarts/jan2009-12th")
IC_from_restart(basepath,expt,"restart011",out_path,"2009-01-01")



Applying Arakawa C grid variable mapping, which is u-velocity on xq, yh; v-velocity on xh, yq; and tracers on xh, yh.


Setting up Initial Conditions
Regridding Velocities... Done.
Regridding Tracers... Done.
Regridding Free surface... Done.


FileNotFoundError: Vertical grid /g/data/x77/ab8992/nino-34/restarts/jan2009-12th/vgrid.nc not found. Make sure `vgrid.nc` exists in /g/data/x77/ab8992/nino-34/restarts/jan2009-12th directory, or pass in a VGrid object via `vgrid_type`.

In [ ]:
import xarray as xr
import numpy as np
import os
from pathlib import Path

def perterb_ic(basepath,expt,restart,out_path,n = 5):
    rundir = basepath / expt
    files = (basepath / expt / "archive" / restart).glob(pattern="access-om3.mom6.r.*")

    # Find the file containing u and v

    for f in files:
        with xr.open_dataset(str(f)) as d:
            if "u" in d.data_vars:
                ufile = f
            if "v" in d.data_vars:
                vfile = f

    # Now in the out path, make a perturbed version of this IC. Make symlinks to all the other files
    u = xr.open_dataset(ufile)
    v = xr.open_dataset(vfile)
    for i in range(n):
        u_p = u
        u_p["u"] += 1e-03 * np.random.random(size = u["u"].shape)
        u_p.attrs = u.attrs
        u_p.to_netcdf(out_path / ufile.name,mode = "w")
    return u["u"]

basepath = Path("/g/data/gb02/ab8992/rom3-rundirs/")
expt = "nino-12th"
out_path = Path("/g/data/x77/ab8992/nino-34/restarts/jan2009-12th")
d = perterb_ic(basepath,expt,"restart011",out_path,1)

In [19]:
import numpy as np
print(d[0,0,0,0].values)
print((d + 1e-03 * np.random.random(size = d.shape)).values[0,0,0,0])

-0.4018673387345914
-0.4016574704460671
